In [2]:
# Agent代理，为其配备不同类型的工具
# 导包
from langchain_ollama import ChatOllama
import warnings
warnings.filterwarnings("ignore")

In [3]:
# 初始化语言模型
llm = ChatOllama(
    model="qwen3.5:2b",
    temperature=0.0, # 消除随机性
    reasoning=False
)

# llm = ChatOllama(
#     model="qwen3:30b",
#     temperature=0.0,
#     reasoning=False,
#     keep_alive="30m",   # 30 分钟内不卸载模型
# )

In [16]:
from langchain_experimental.agents.agent_toolkits import create_python_agent
from langchain_classic.agents import load_tools, initialize_agent
from langchain_classic.agents import AgentType
from langchain_experimental.tools.python.tool import PythonAstREPLTool
from langchain_experimental.utilities.python import PythonREPL

In [60]:
# 加载llm-math和wikipedia两个工具
tools = load_tools(["llm-math","wikipedia"], llm=llm)

In [62]:
# 初始化一个agent
agent = initialize_agent(
    tools,
    llm,
    # CHAT指向了其专为与Chat模型一起工作而优化的Agent
    # REACT是一种组织Prompt的技术，能最大化语言模型的推理能力
    agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True,
    # 在语言模型输出的内容无法被正常解析时起帮助：遇到内容无法被正常解析时，将格式错误的内容传回语言模型，并要求他自行纠正
    verbose=True
)

In [61]:
agent("What is the 25% of 300？")

[chain/start] [chain:AgentExecutor] Entering Chain run with input:
{
  "input": "What is the 25% of 300？"
}
[chain/start] [chain:AgentExecutor > chain:LLMChain] Entering Chain run with input:
{
  "input": "What is the 25% of 300？",
  "agent_scratchpad": "",
  "stop": [
    "\nObservation:",
    "\n\tObservation:"
  ]
}
[llm/start] [chain:AgentExecutor > chain:LLMChain > llm:ChatOllama] Entering LLM run with input:
{
  "prompts": [
    "Human: You are an agent designed to write and execute python code to answer questions.\nYou have access to a python REPL, which you can use to execute python code.\nIf you get an error, debug your code and try again.\nOnly use the output of your code to answer the question. \nYou might know the answer without running any code, but you should still run the code to get the answer.\nIf it does not seem like you can write code to answer the question, just return \"I don't know\" as the answer.\n\n\npython_repl_ast - A Python shell. Use this to execute python

{'input': 'What is the 25% of 300？', 'output': '75.0'}

In [29]:
# 使用维基百科的API，此处需要挂代理
question = "Tom M. Mitchell is an American computer scientist \
and the Founders University Professor at Carnegie Mellon University (CMU)\
what book did he write?"
result = agent(question)



> Entering new AgentExecutor chain...
Thought: The user is asking for the books written by Tom M. Mitchell. I have access to the Wikipedia tool, which can provide general information about people, places, companies, facts, and historical events. I will search for Tom M. Mitchell on Wikipedia to find a list of his works.

Action:
```
{
  "action": "wikipedia",
  "action_input": "Tom M. Mitchell"
}
```
Observation: Page: Tom M. Mitchell
Summary: Tom Michael Mitchell (born August 9, 1951) is an American computer scientist and the Founders University Professor at Carnegie Mellon University (CMU). He is a founder and former chair of the Machine Learning Department at CMU. Mitchell is known for his contributions to the advancement of machine learning, artificial intelligence, and cognitive neuroscience and is the author of the textbook Machine Learning. He is a member of the United States National Academy of Engineering since 2010. He is also a Fellow of the American Academy of Arts and Sc

In [63]:
# 类似于ChatGpt和Github Copilot，通过使用语言模型写代码，然后执行生成的代码
agent = create_python_agent(
    llm,
    tool=PythonAstREPLTool(), # REPLTool类似于Jupyter,Agent可以利用其执行代码
    verbose=True
)

# 让agent给一组客户名单排序
customer_list = [["Harrison", "Chase"],
                 ["Lang", "Chain"],
                 ["Dolly", "Too"],
                 ["Elle", "Elem"],
                 ["Geoff", "Fusion"],
                 ["Trance", "Former"],
                 ["Jen", "Ayai"]
                ]

agent.run(f"""
    Sort these customers by \ last name and then first name \ and print the output:{customer_list}
""")

[chain/start] [chain:AgentExecutor] Entering Chain run with input:
{
  "input": "\n    Sort these customers by \\ last name and then first name \\ and print the output:[['Harrison', 'Chase'], ['Lang', 'Chain'], ['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Trance', 'Former'], ['Jen', 'Ayai']]\n"
}
[chain/start] [chain:AgentExecutor > chain:LLMChain] Entering Chain run with input:
{
  "input": "\n    Sort these customers by \\ last name and then first name \\ and print the output:[['Harrison', 'Chase'], ['Lang', 'Chain'], ['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Trance', 'Former'], ['Jen', 'Ayai']]\n",
  "agent_scratchpad": "",
  "stop": [
    "\nObservation:",
    "\n\tObservation:"
  ]
}
[llm/start] [chain:AgentExecutor > chain:LLMChain > llm:ChatOllama] Entering LLM run with input:
{
  "prompts": [
    "Human: You are an agent designed to write and execute python code to answer questions.\nYou have access to a python REPL, which you can use to execute p

"The sorted list of customers is:\n1. ['Jen', 'Ayai']\n2. ['Lang', 'Chain']\n3. ['Harrison', 'Chase']\n4. ['Elle', 'Elem']\n5. ['Trance', 'Former']\n6. ['Geoff', 'Fusion']\n7. ['Dolly', 'Too']"

In [64]:
from langchain_classic.globals import set_debug
set_debug(True)

agent.invoke(f"""
Sort these customers by \ last name and then first name \ and print the output:{customer_list}
""")

[chain/start] [chain:AgentExecutor] Entering Chain run with input:
{
  "input": "\nSort these customers by \\ last name and then first name \\ and print the output:[['Harrison', 'Chase'], ['Lang', 'Chain'], ['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Trance', 'Former'], ['Jen', 'Ayai']]\n"
}
[chain/start] [chain:AgentExecutor > chain:LLMChain] Entering Chain run with input:
{
  "input": "\nSort these customers by \\ last name and then first name \\ and print the output:[['Harrison', 'Chase'], ['Lang', 'Chain'], ['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Trance', 'Former'], ['Jen', 'Ayai']]\n",
  "agent_scratchpad": "",
  "stop": [
    "\nObservation:",
    "\n\tObservation:"
  ]
}
[llm/start] [chain:AgentExecutor > chain:LLMChain > llm:ChatOllama] Entering LLM run with input:
{
  "prompts": [
    "Human: You are an agent designed to write and execute python code to answer questions.\nYou have access to a python REPL, which you can use to execute python co

{'input': "\nSort these customers by \\ last name and then first name \\ and print the output:[['Harrison', 'Chase'], ['Lang', 'Chain'], ['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Trance', 'Former'], ['Jen', 'Ayai']]\n",
 'output': "[['Jen', 'Ayai'], ['Lang', 'Chain'], ['Harrison', 'Chase'], ['Elle', 'Elem'], ['Trance', 'Former'], ['Geoff', 'Fusion'], ['Dolly', 'Too']]"}

In [65]:
# 创建自定义工具
from langchain_core.tools import tool
from datetime import date

In [74]:
# 导入修饰器
@tool
def time(text: str) -> str:
    """Returns todays date, use this for any \
    questions related to knowing todays date. \
    The input should always be an empty string, \
    and this function will always return todays \
    date - any date mathmatics should occur \
    outside this function."""
    return str(date.today())



In [78]:
# 创建新的代理
agent = initialize_agent(
    tools + [time],
    llm,
    agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True,
    verbose=True
)

In [84]:
agent.invoke("What day is it tommorrow?")
# 想要正常输出需要更换参数更大的模型

[chain/start] [chain:AgentExecutor] Entering Chain run with input:
{
  "input": "What day is it tommorrow?"
}
[chain/start] [chain:AgentExecutor > chain:LLMChain] Entering Chain run with input:
{
  "input": "What day is it tommorrow?",
  "agent_scratchpad": "",
  "stop": [
    "Observation:"
  ]
}
[llm/start] [chain:AgentExecutor > chain:LLMChain > llm:ChatOllama] Entering LLM run with input:
{
  "prompts": [
    "System: Answer the following questions as best you can. You have access to the following tools:\n\nCalculator: Useful for when you need to answer questions about math.\nwikipedia: A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.\ntime: Returns todays date, use this for any     questions related to knowing todays date.     The input should always be an empty string,     and this function will always return todays     date - any date math

ValueError: LLMMathChain._evaluate("
2026-09-04 + 1 day
") raised error: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (<expr>, line 1). Please try again with a valid numerical expression